In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT/CMS style
hep.style.use(hep.style.ROOT)

def get_integrated_eff(pass_mask, total_mask):
    """ Calculates the overall integrated efficiency and binomial error as a percentage """
    n_tot = np.sum(total_mask)
    n_pass = np.sum(total_mask & pass_mask)
    
    if n_tot == 0: 
        return 0.0, 0.0
        
    eff = n_pass / n_tot
    err = np.sqrt(eff * (1.0 - eff) / n_tot)
    
    return eff * 100.0, err * 100.0

def calc_efficiency(variable, pass_mask, total_mask, bins):
    """ Calculates bin-by-bin efficiency """
    var_total = variable[total_mask]
    var_pass  = variable[total_mask & pass_mask]
    
    counts_total, _ = np.histogram(var_total, bins=bins)
    counts_pass, _  = np.histogram(var_pass, bins=bins)
    
    counts_total = np.asarray(counts_total, dtype=float)
    counts_pass  = np.asarray(counts_pass, dtype=float)
    
    eff = np.zeros_like(counts_total)
    valid_bins = counts_total > 0
    eff[valid_bins] = counts_pass[valid_bins] / counts_total[valid_bins]
    
    err = np.zeros_like(counts_total)
    err[valid_bins] = np.sqrt(eff[valid_bins] * (1.0 - eff[valid_bins]) / counts_total[valid_bins])
    
    return eff, err

def plot_eff_panel(ax, variable, mask_acc, mask_reco, mask_gen, bins, xlabel):
    # 1. Calculate bin-by-bin curves
    acc_eff, acc_err = calc_efficiency(variable, pass_mask=mask_acc, total_mask=mask_gen, bins=bins)
    reco_eff, reco_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_acc, bins=bins)
    tot_eff, tot_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_gen, bins=bins)
    
    # 2. Calculate integrated overall averages
    acc_avg, acc_avg_err   = get_integrated_eff(mask_acc, mask_gen)
    reco_avg, reco_avg_err = get_integrated_eff(mask_reco, mask_acc)
    tot_avg, tot_avg_err   = get_integrated_eff(mask_reco, mask_gen)
    
    # 3. Format the legend labels (Double braces {{ }} are used to escape LaTeX in f-strings)
    l_acc  = fr'Acceptance ($A$) [{acc_avg:.1f} $\pm$ {acc_avg_err:.1f}%]'
    l_reco = fr'Reco Eff. ($\epsilon_{{reco}}$) [{reco_avg:.1f} $\pm$ {reco_avg_err:.1f}%]'
    l_tot  = fr'Total Eff. [{tot_avg:.1f} $\pm$ {tot_avg_err:.1f}%]'
    
    # 4. Plot
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    ax.errorbar(bin_centers, acc_eff, yerr=acc_err, fmt='o-', color='royalblue', label=l_acc, markersize=5)
    ax.errorbar(bin_centers, reco_eff, yerr=reco_err, fmt='s-', color='darkorange', label=l_reco, markersize=5)
    ax.errorbar(bin_centers, tot_eff, yerr=tot_err, fmt='^-', color='black', label=l_tot, markersize=5)
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Efficiency")
    ax.set_ylim(0.0, 1.25) # Slightly expanded y-limit to give the wider legend room to breathe
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='lower right', fontsize=10)

def main():
    filename = "~/merged_elecs_test_tuples.root" 
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    branches = [
        "true_mass", "true_pT", "true_y", "true_W", "true_t",
        "true_ePlus_eta", "true_decay_eMinus_eta",
        "reco_mass", "reco_W", "sigma_epz"
    ]
    
    print("Pulling arrays from RDataFrame into NumPy...")
    data = df.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE UPSILON STATES TO LOOP OVER
    # =========================================================================
    upsilon_states = {
        "1S": {"min_mass": 9.0,  "max_mass": 9.8,  "title": r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$"},
        "2S": {"min_mass": 9.8,  "max_mass": 10.2, "title": r"$\Upsilon(2S) \rightarrow e^{+}e^{-}$"},
        "3S": {"min_mass": 10.2, "max_mass": 10.6, "title": r"$\Upsilon(3S) \rightarrow e^{+}e^{-}$"}
    }

    # =========================================================================
    # GLOBAL CUTS (Apply to all states)
    # =========================================================================
    in_acc = (np.abs(data["true_ePlus_eta"]) < 3.5) & (np.abs(data["true_decay_eMinus_eta"]) < 3.5)
    
    passed_reco = (data["reco_mass"] >= 8.0) & (data["reco_mass"] <= 12.0) \
                  & (np.abs(data["sigma_epz"] - 36.0) <= 3.6) & (data["reco_W"] >= 20.0)

    # =========================================================================
    # GENERATE PLOTS FOR EACH STATE
    # =========================================================================
    for state_name, config in upsilon_states.items():
        print(f"Calculating efficiencies for {state_name}...")
        
        mask_state = (data["true_mass"] > config["min_mass"]) & (data["true_mass"] <= config["max_mass"])
        
        if np.sum(mask_state) == 0:
            print(f"  -> No events found for {state_name} in this file. Skipping.")
            continue

        mask_gen_state  = mask_state
        mask_acc_state  = mask_state & in_acc
        mask_reco_state = mask_state & passed_reco

        fig, axs = plt.subplots(2, 2, figsize=(16, 14))
        
        pt_bins = np.linspace(0.0, 2.5, 25)
        plot_eff_panel(axs[0, 0], data["true_pT"], mask_acc_state, mask_reco_state, mask_gen_state, pt_bins, r"True Upsilon $p_T$ [GeV/c]")
        
        y_bins = np.linspace(-4.0, 4.0, 40)
        plot_eff_panel(axs[0, 1], data["true_y"], mask_acc_state, mask_reco_state, mask_gen_state, y_bins, r"True Upsilon Rapidity $y$")
        axs[0, 1].set_title(f"{config['title']} Efficiency", loc="right", fontsize=14)
        hep.cms.text("Simulation Preliminary", loc=0, ax=axs[0, 1])

        w_bins = np.linspace(20, 150, 26)
        plot_eff_panel(axs[1, 0], data["true_W"], mask_acc_state, mask_reco_state, mask_gen_state, w_bins, r"True $W_{\gamma p}$ [GeV]")
        
        t_bins = np.linspace(0.0, 1.5, 15)
        plot_eff_panel(axs[1, 1], data["true_t"], mask_acc_state, mask_reco_state, mask_gen_state, t_bins, r"True $|t|$ [GeV$^2$]")

        fig.tight_layout()
        output_filename = f"Upsilon_{state_name}_Kinematic_Efficiencies.png"
        fig.savefig(output_filename, dpi=300)
        print(f"  -> Saved {output_filename}")
        plt.close(fig) 

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style
hep.style.use(hep.style.ROOT)

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=13, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')
    
    # 3. Reaction
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')
            
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')

def get_integrated_eff(pass_mask, total_mask):
    """ Calculates the overall integrated efficiency and binomial error as a percentage """
    n_tot = np.sum(total_mask)
    n_pass = np.sum(total_mask & pass_mask)
    
    if n_tot == 0: 
        return 0.0, 0.0
        
    eff = n_pass / n_tot
    err = np.sqrt(eff * (1.0 - eff) / n_tot)
    
    return eff * 100.0, err * 100.0

def calc_efficiency(variable, pass_mask, total_mask, bins):
    """ Calculates bin-by-bin efficiency """
    var_total = variable[total_mask]
    var_pass  = variable[total_mask & pass_mask]
    
    counts_total, _ = np.histogram(var_total, bins=bins)
    counts_pass, _  = np.histogram(var_pass, bins=bins)
    
    counts_total = np.asarray(counts_total, dtype=float)
    counts_pass  = np.asarray(counts_pass, dtype=float)
    
    eff = np.zeros_like(counts_total)
    valid_bins = counts_total > 0
    eff[valid_bins] = counts_pass[valid_bins] / counts_total[valid_bins]
    
    err = np.zeros_like(counts_total)
    err[valid_bins] = np.sqrt(eff[valid_bins] * (1.0 - eff[valid_bins]) / counts_total[valid_bins])
    
    return eff, err

def plot_eff_panel(ax, variable, mask_acc, mask_reco, mask_gen, bins, xlabel):
    # 1. Calculate bin-by-bin curves
    acc_eff, acc_err = calc_efficiency(variable, pass_mask=mask_acc, total_mask=mask_gen, bins=bins)
    reco_eff, reco_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_acc, bins=bins)
    tot_eff, tot_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_gen, bins=bins)
    
    # 2. Calculate integrated overall averages
    acc_avg, acc_avg_err   = get_integrated_eff(mask_acc, mask_gen)
    reco_avg, reco_avg_err = get_integrated_eff(mask_reco, mask_acc)
    tot_avg, tot_avg_err   = get_integrated_eff(mask_reco, mask_gen)
    
    # 3. Format the legend labels (Double braces {{ }} are used to escape LaTeX in f-strings)
    l_acc  = fr'Acceptance ($A$) [{acc_avg:.1f} $\pm$ {acc_avg_err:.1f}%]'
    l_reco = fr'Reco Eff. ($\epsilon_{{reco}}$) [{reco_avg:.1f} $\pm$ {reco_avg_err:.1f}%]'
    l_tot  = fr'Total Eff. [{tot_avg:.1f} $\pm$ {tot_avg_err:.1f}%]'
    
    # 4. Plot
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    ax.errorbar(bin_centers, acc_eff, yerr=acc_err, fmt='o-', color='royalblue', label=l_acc, markersize=5)
    ax.errorbar(bin_centers, reco_eff, yerr=reco_err, fmt='s-', color='darkorange', label=l_reco, markersize=5)
    ax.errorbar(bin_centers, tot_eff, yerr=tot_err, fmt='^-', color='black', label=l_tot, markersize=5)
    
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("Efficiency", fontsize=16)
    
    # Expand Y-axis up to 1.45 to ensure the ePIC text block fits cleanly above the data
    ax.set_ylim(0.0, 1.45) 
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    
    # Add Standard ePIC formatting
    draw_epic_labels(ax)
    
    ax.legend(loc='lower right', frameon=False, fontsize=12)

def main():
    filename = "~/merged_elecs_test_tuples_old.root" 
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    branches = [
        "true_mass", "true_pT", "true_y", "true_W", "true_t",
        "true_ePlus_eta", "true_decay_eMinus_eta",
        "reco_mass", "reco_W", "sigma_epz"
    ]
    
    print("Pulling arrays from RDataFrame into NumPy...")
    data = df.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE UPSILON STATES TO LOOP OVER
    # =========================================================================
    upsilon_states = {
        "1S": {"min_mass": 9.0,  "max_mass": 9.8,  "title": r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$"},
        "2S": {"min_mass": 9.8,  "max_mass": 10.2, "title": r"$\Upsilon(2S) \rightarrow e^{+}e^{-}$"},
        "3S": {"min_mass": 10.2, "max_mass": 10.6, "title": r"$\Upsilon(3S) \rightarrow e^{+}e^{-}$"}
    }

    # =========================================================================
    # GLOBAL CUTS (Apply to all states)
    # =========================================================================
    # Tracking Acceptance Cut
    in_acc = (np.abs(data["true_ePlus_eta"]) < 3.5) & (np.abs(data["true_decay_eMinus_eta"]) < 3.5)
    
    # Full Reconstruction / Selection Cuts
    #passed_reco = (data["reco_mass"] >= 8.0) & (data["reco_mass"] <= 12.0) \
    #              & (np.abs(data["sigma_epz"] - 36.0) <= 3.6) & (data["reco_W"] >= 20.0)
    passed_reco = (data["reco_mass"] >= 8.0) & (data["reco_mass"] <= 12.0) \
                  & (data["reco_W"] >= 0.0)

    # =========================================================================
    # GENERATE PLOTS FOR EACH STATE
    # =========================================================================
    for state_name, config in upsilon_states.items():
        print(f"Calculating efficiencies for {state_name}...")
        
        mask_state = (data["true_mass"] > config["min_mass"]) & (data["true_mass"] <= config["max_mass"])
        
        if np.sum(mask_state) == 0:
            print(f"  -> No events found for {state_name} in this file. Skipping.")
            continue

        mask_gen_state  = mask_state
        mask_acc_state  = mask_state & in_acc
        mask_reco_state = mask_state & passed_reco

        fig, axs = plt.subplots(2, 2, figsize=(16, 14))
        
        # Super title for the specific Upsilon State
        fig.suptitle(f"{config['title']} | Kinematic Efficiencies", 
                     fontsize=24, fontweight='bold', y=0.96)
        
        pt_bins = np.linspace(0.0, 2.5, 25)
        plot_eff_panel(axs[0, 0], data["true_pT"], mask_acc_state, mask_reco_state, mask_gen_state, pt_bins, r"True Upsilon $p_T$ [GeV/c]")
        
        y_bins = np.linspace(-4.0, 4.0, 40)
        plot_eff_panel(axs[0, 1], data["true_y"], mask_acc_state, mask_reco_state, mask_gen_state, y_bins, r"True Upsilon Rapidity $y$")

        w_bins = np.linspace(20, 150, 26)
        plot_eff_panel(axs[1, 0], data["true_W"], mask_acc_state, mask_reco_state, mask_gen_state, w_bins, r"True $W_{\gamma p}$ [GeV]")
        
        t_bins = np.linspace(0.0, 1.5, 15)
        plot_eff_panel(axs[1, 1], data["true_t"], mask_acc_state, mask_reco_state, mask_gen_state, t_bins, r"True $|t|$ [GeV$^2$]")

        # Clean layout
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        output_filename = f"Upsilon_{state_name}_Kinematic_Efficiencies.png"
        fig.savefig(output_filename, dpi=300)
        print(f"  -> Saved {output_filename}")
        plt.close(fig) 

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style
hep.style.use(hep.style.ROOT)

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=13, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')
    
    # 3. Reaction
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')
            
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=12, ha='left', va='top')

def get_integrated_eff(pass_mask, total_mask):
    """ Calculates the overall integrated efficiency and binomial error as a percentage """
    n_tot = np.sum(total_mask)
    n_pass = np.sum(total_mask & pass_mask)
    
    if n_tot == 0: 
        return 0.0, 0.0
        
    eff = n_pass / n_tot
    err = np.sqrt(eff * (1.0 - eff) / n_tot)
    
    return eff * 100.0, err * 100.0

def calc_efficiency(variable, pass_mask, total_mask, bins):
    """ Calculates bin-by-bin efficiency """
    var_total = variable[total_mask]
    var_pass  = variable[total_mask & pass_mask]
    
    counts_total, _ = np.histogram(var_total, bins=bins)
    counts_pass, _  = np.histogram(var_pass, bins=bins)
    
    counts_total = np.asarray(counts_total, dtype=float)
    counts_pass  = np.asarray(counts_pass, dtype=float)
    
    # Initialize with NaN instead of 0.0 to prevent plunging lines in matplotlib!
    eff = np.full_like(counts_total, np.nan, dtype=float)
    err = np.full_like(counts_total, np.nan, dtype=float)
    
    valid_bins = counts_total > 0
    eff[valid_bins] = counts_pass[valid_bins] / counts_total[valid_bins]
    err[valid_bins] = np.sqrt(eff[valid_bins] * (1.0 - eff[valid_bins]) / counts_total[valid_bins])
    
    return eff, err

def plot_eff_panel(ax, variable, mask_acc, mask_reco, mask_gen, bins, xlabel):
    # 1. Calculate bin-by-bin curves
    acc_eff, acc_err = calc_efficiency(variable, pass_mask=mask_acc, total_mask=mask_gen, bins=bins)
    reco_eff, reco_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_acc, bins=bins)
    tot_eff, tot_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_gen, bins=bins)
    
    # 2. Calculate integrated overall averages
    acc_avg, acc_avg_err   = get_integrated_eff(mask_acc, mask_gen)
    reco_avg, reco_avg_err = get_integrated_eff(mask_reco, mask_acc)
    tot_avg, tot_avg_err   = get_integrated_eff(mask_reco, mask_gen)
    
    # 3. Format the legend labels
    l_acc  = fr'Acceptance ($A$) [{acc_avg:.1f} $\pm$ {acc_avg_err:.1f}%]'
    l_reco = fr'Reco Eff. ($\epsilon_{{reco}}$) [{reco_avg:.1f} $\pm$ {reco_avg_err:.1f}%]'
    l_tot  = fr'Total Eff. [{tot_avg:.1f} $\pm$ {tot_avg_err:.1f}%]'
    
    # 4. Plot
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    ax.errorbar(bin_centers, acc_eff, yerr=acc_err, fmt='o-', color='royalblue', label=l_acc, markersize=5)
    ax.errorbar(bin_centers, reco_eff, yerr=reco_err, fmt='s-', color='darkorange', label=l_reco, markersize=5)
    ax.errorbar(bin_centers, tot_eff, yerr=tot_err, fmt='^-', color='black', label=l_tot, markersize=5)
    
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("Efficiency", fontsize=16)
    
    # Expand Y-axis to ensure ePIC text block fits above data
    ax.set_ylim(0.0, 1.45) 
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    
    # Add Standard ePIC formatting
    draw_epic_labels(ax)
    
    ax.legend(loc='lower right', frameon=False, fontsize=12)

def main():
    filename = "~/merged_elecs_test_10t100.root" 
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # Including the specific decay lepton branches
    branches = [
        "true_mass", 
        "true_ePlus_pT", "true_ePlus_eta",
        "true_decay_eMinus_pT", "true_decay_eMinus_eta",
        "reco_mass", "reco_W", "sigma_epz"
    ]
    
    print("Pulling arrays from RDataFrame into NumPy...")
    data = df.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE UPSILON STATES TO LOOP OVER
    # =========================================================================
    upsilon_states = {
        "1S": {"min_mass": 9.0,  "max_mass": 9.8,  "title": r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$"},
        "2S": {"min_mass": 9.8,  "max_mass": 10.2, "title": r"$\Upsilon(2S) \rightarrow e^{+}e^{-}$"},
        "3S": {"min_mass": 10.2, "max_mass": 10.6, "title": r"$\Upsilon(3S) \rightarrow e^{+}e^{-}$"}
    }

    # =========================================================================
    # GLOBAL CUTS (Apply to all states)
    # =========================================================================
    # Tracking Acceptance Cut
    in_acc = (np.abs(data["true_ePlus_eta"]) < 3.5) & (np.abs(data["true_decay_eMinus_eta"]) < 3.5)
    
    # Full Reconstruction / Selection Cuts
    #passed_reco = (data["reco_mass"] >= 8.0) & (data["reco_mass"] <= 12.0) \
    #              & (np.abs(data["sigma_epz"] - 36.0) <= 3.6) & (data["reco_W"] >= 20.0)
    passed_reco = (data["reco_mass"] >= 8.0) & (data["reco_mass"] <= 12.0) \
                   & (data["reco_W"] >= 20.0)
    # =========================================================================
    # GENERATE PLOTS FOR EACH STATE
    # =========================================================================
    for state_name, config in upsilon_states.items():
        print(f"Calculating efficiencies for {state_name}...")
        
        mask_state = (data["true_mass"] > config["min_mass"]) & (data["true_mass"] <= config["max_mass"])
        
        if np.sum(mask_state) == 0:
            print(f"  -> No events found for {state_name} in this file. Skipping.")
            continue

        mask_gen_state  = mask_state
        mask_acc_state  = mask_state & in_acc
        mask_reco_state = mask_state & passed_reco

        fig, axs = plt.subplots(2, 2, figsize=(16, 14))
        
        # Super title for the specific Upsilon State
        fig.suptitle(f"{config['title']} | Decay Lepton Efficiencies", 
                     fontsize=24, fontweight='bold', y=0.96)
        
        # Standard tracking pT binning
        pt_bins = np.linspace(0.0, 10.0, 40)
        # Standard tracking eta binning
        eta_bins = np.linspace(-4.0, 4.0, 40)
        
        # Top Row: Decay e- 
        plot_eff_panel(axs[0, 0], data["true_decay_eMinus_pT"], mask_acc_state, mask_reco_state, mask_gen_state, pt_bins, r"True Decay $e^-$ $p_T$ [GeV/c]")
        plot_eff_panel(axs[0, 1], data["true_decay_eMinus_eta"], mask_acc_state, mask_reco_state, mask_gen_state, eta_bins, r"True Decay $e^-$ $\eta$")

        # Bottom Row: Decay e+
        plot_eff_panel(axs[1, 0], data["true_ePlus_pT"], mask_acc_state, mask_reco_state, mask_gen_state, pt_bins, r"True Decay $e^+$ $p_T$ [GeV/c]")
        plot_eff_panel(axs[1, 1], data["true_ePlus_eta"], mask_acc_state, mask_reco_state, mask_gen_state, eta_bins, r"True Decay $e^+$ $\eta$")

        # Clean layout
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        output_filename = f"Upsilon_{state_name}_DecayLepton_Efficiencies.png"
        fig.savefig(output_filename, dpi=300)
        print(f"  -> Saved {output_filename}")
        plt.close(fig) 

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Overlay kinematic efficiencies for two beam energies in each panel.

The layout follows the existing efficiency script:
  - one 2x2 figure per Υ(nS) state
  - panels versus true Υ pT, true Υ rapidity, true W_gamma p, and true |t|

Each panel overlays the 10x100 and 18x275 samples using energy-specific
exclusivity selections:
  10x100:  |Sigma(E-pz) - 20| <= 2 GeV
  18x275:  |Sigma(E-pz) - 36| <= 3.6 GeV

No event weights or branching-ratio corrections are used in the efficiency
calculation. Efficiencies are event-count ratios:
  A        = N(acceptance) / N(generated)
  eps_reco = N(reconstructed selection) / N(acceptance)
  total    = N(reconstructed selection) / N(generated)
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

TREE_NAME = "UpsilonTree"

SAMPLES = {
    "low": {
        "file": "~/merged_elecs_test_10t100_10x100.root",
        # Alternative common EOS-style name:
        # "file": "/eos/user/g/gkrintir/merged_elecs_test_10t100.root",
        "label": "10x100 GeV",
        "campaign": "26.04.1 Campaign",
        "electron_energy": 10.0,
        "sqrt_s": 63.0,
        "color": "dodgerblue",
    },
    "high": {
        "file": "~/merged_elecs_test_tuples_fix2.root",
        # Alternative common EOS-style name:
        # "file": "/eos/user/g/gkrintir/merged_elecs_test_tuples_fix.root",
        "label": "18x275 GeV",
        "campaign": "26.03.1 Campaign",
        "electron_energy": 18.0,
        "sqrt_s": 141.0,
        "color": "darkorange",
    },
}

APPLY_SIGMA_EPZ_CUT = True
SIGMA_EPZ_HALF_WIDTH_FRACTION = 0.20

# Same broad truth-state windows as the original efficiency script.
UPSILON_STATES = {
    "1S": {"min_mass": 9.0,  "max_mass": 9.8,  "title": r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$"},
    "2S": {"min_mass": 9.8,  "max_mass": 10.2, "title": r"$\Upsilon(2S) \rightarrow e^{+}e^{-}$"},
    "3S": {"min_mass": 10.2, "max_mass": 10.6, "title": r"$\Upsilon(3S) \rightarrow e^{+}e^{-}$"},
}


def sigma_epz_info(sample_key):
    cfg = SAMPLES[sample_key]
    center = 2.0 * cfg["electron_energy"]
    half_width = SIGMA_EPZ_HALF_WIDTH_FRACTION * cfg["electron_energy"]
    return center, half_width


def draw_epic_labels(ax, state_title):
    """Draw comparison labels in each panel."""
    low_center, low_hw = sigma_epz_info("low")
    high_center, high_hw = sigma_epz_info("high")

    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, state_title + " efficiency comparison",
            transform=ax.transAxes, fontsize=10.5, ha="left", va="top")
    ax.text(0.04, 0.82,
            rf"{SAMPLES['low']['campaign']}, {SAMPLES['low']['label']}: "
            rf"$|\Sigma(E-p_z)-{low_center:g}|<{low_hw:g}$ GeV",
            transform=ax.transAxes, fontsize=9.5, color=SAMPLES["low"]["color"], ha="left", va="top")
    ax.text(0.04, 0.77,
            rf"{SAMPLES['high']['campaign']}, {SAMPLES['high']['label']}: "
            rf"$|\Sigma(E-p_z)-{high_center:g}|<{high_hw:g}$ GeV",
            transform=ax.transAxes, fontsize=9.5, color=SAMPLES["high"]["color"], ha="left", va="top")


def get_integrated_eff(pass_mask, total_mask):
    n_tot = np.sum(total_mask)
    n_pass = np.sum(total_mask & pass_mask)

    if n_tot == 0:
        return 0.0, 0.0

    eff = n_pass / n_tot
    err = np.sqrt(eff * (1.0 - eff) / n_tot)
    return eff * 100.0, err * 100.0


def calc_efficiency(variable, pass_mask, total_mask, bins):
    var_total = variable[total_mask]
    var_pass = variable[total_mask & pass_mask]

    counts_total, _ = np.histogram(var_total, bins=bins)
    counts_pass, _ = np.histogram(var_pass, bins=bins)

    counts_total = np.asarray(counts_total, dtype=float)
    counts_pass = np.asarray(counts_pass, dtype=float)

    eff = np.zeros_like(counts_total)
    valid = counts_total > 0
    eff[valid] = counts_pass[valid] / counts_total[valid]

    err = np.zeros_like(counts_total)
    err[valid] = np.sqrt(eff[valid] * (1.0 - eff[valid]) / counts_total[valid])
    return eff, err


def build_masks(data, state_cfg, sample_key):
    mask_state = (
        (data["true_mass"] > state_cfg["min_mass"]) &
        (data["true_mass"] <= state_cfg["max_mass"])
    )

    in_acc = (
        (np.abs(data["true_ePlus_eta"]) < 3.5) &
        (np.abs(data["true_decay_eMinus_eta"]) < 3.5)
    )

    passed_reco = (
        (data["reco_mass"] >= 8.0) &
        (data["reco_mass"] <= 12.0) &
        (data["reco_W"] >= 0.0)
    )

    if APPLY_SIGMA_EPZ_CUT:
        center, half_width = sigma_epz_info(sample_key)
        passed_reco = passed_reco & (np.abs(data["sigma_epz"] - center) <= half_width)

    mask_gen_state = mask_state
    mask_acc_state = mask_state & in_acc
    mask_reco_state = mask_state & passed_reco

    return mask_gen_state, mask_acc_state, mask_reco_state


def plot_eff_panel(ax, sample_data, variable_name, bins, xlabel, state_cfg):
    """Overlay efficiencies for low and high samples in one panel."""
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    markers = {
        "acc": "o",
        "reco": "s",
        "tot": "^",
    }

    linestyles = {
        "acc": "-",
        "reco": "--",
        "tot": ":",
    }

    for sample_key in ["low", "high"]:
        data = sample_data[sample_key]
        cfg = SAMPLES[sample_key]

        mask_gen, mask_acc, mask_reco = build_masks(data, state_cfg, sample_key)
        variable = data[variable_name]

        acc_eff, acc_err = calc_efficiency(variable, pass_mask=mask_acc, total_mask=mask_gen, bins=bins)
        reco_eff, reco_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_acc, bins=bins)
        tot_eff, tot_err = calc_efficiency(variable, pass_mask=mask_reco, total_mask=mask_gen, bins=bins)

        acc_avg, acc_avg_err = get_integrated_eff(mask_acc, mask_gen)
        reco_avg, reco_avg_err = get_integrated_eff(mask_reco, mask_acc)
        tot_avg, tot_avg_err = get_integrated_eff(mask_reco, mask_gen)

        sample_label = cfg["label"]

        ax.errorbar(
            bin_centers, acc_eff, yerr=acc_err,
            fmt=markers["acc"] + linestyles["acc"],
            color=cfg["color"], markersize=4,
            label=rf"{sample_label} $A$ [{acc_avg:.1f}$\pm${acc_avg_err:.1f}\%]"
        )
        ax.errorbar(
            bin_centers, reco_eff, yerr=reco_err,
            fmt=markers["reco"] + linestyles["reco"],
            color=cfg["color"], markersize=4,
            label=rf"{sample_label} $\epsilon_{{reco}}$ [{reco_avg:.1f}$\pm${reco_avg_err:.1f}\%]"
        )
        ax.errorbar(
            bin_centers, tot_eff, yerr=tot_err,
            fmt=markers["tot"] + linestyles["tot"],
            color=cfg["color"], markersize=4,
            label=rf"{sample_label} total [{tot_avg:.1f}$\pm${tot_avg_err:.1f}\%]"
        )

    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel("Efficiency", fontsize=15)
    ax.set_ylim(0.0, 1.45)
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)

    draw_epic_labels(ax, state_cfg["title"])

    ax.legend(loc="lower right", frameon=False, fontsize=8.5, ncol=1)


def load_sample(sample_key):
    cfg = SAMPLES[sample_key]
    try:
        df = ROOT.RDataFrame(TREE_NAME, cfg["file"])
    except Exception as e:
        raise RuntimeError(f"Error opening {cfg['file']}: {e}")

    branches = [
        "true_mass", "true_pT", "true_y", "true_W", "true_t",
        "true_ePlus_eta", "true_decay_eMinus_eta",
        "reco_mass", "reco_W", "sigma_epz",
    ]

    print(f"Pulling arrays for {cfg['label']} from {cfg['file']}")
    data = df.AsNumpy(branches)
    return {k: np.asarray(v) for k, v in data.items()}


def main():
    print("\n=== Two-energy kinematic efficiency overlay ===")
    print(f"APPLY_SIGMA_EPZ_CUT = {APPLY_SIGMA_EPZ_CUT}")
    for key in ["low", "high"]:
        c, hw = sigma_epz_info(key)
        print(f"{SAMPLES[key]['label']}: |Sigma(E-pz)-{c:g}| < {hw:g} GeV")

    sample_data = {key: load_sample(key) for key in ["low", "high"]}

    for state_name, state_cfg in UPSILON_STATES.items():
        print(f"Calculating overlaid efficiencies for {state_name}...")

        fig, axs = plt.subplots(2, 2, figsize=(17, 14))
        fig.suptitle(f"{state_cfg['title']} | Kinematic Efficiencies, 10x100 vs 18x275",
                     fontsize=23, fontweight="bold", y=0.96)

        pt_bins = np.linspace(0.0, 2.5, 25)
        plot_eff_panel(
            axs[0, 0], sample_data, "true_pT", pt_bins,
            r"True Upsilon $p_T$ [GeV/c]", state_cfg
        )

        y_bins = np.linspace(-4.0, 4.0, 40)
        plot_eff_panel(
            axs[0, 1], sample_data, "true_y", y_bins,
            r"True Upsilon rapidity $y$", state_cfg
        )

        w_bins = np.linspace(20, 150, 26)
        plot_eff_panel(
            axs[1, 0], sample_data, "true_W", w_bins,
            r"True $W_{\gamma p}$ [GeV]", state_cfg
        )

        t_bins = np.linspace(0.0, 1.5, 15)
        plot_eff_panel(
            axs[1, 1], sample_data, "true_t", t_bins,
            r"True $|t|$ [GeV$^2$]", state_cfg
        )

        fig.tight_layout(rect=[0, 0, 1, 0.94])
        output_filename = f"Upsilon_{state_name}_Kinematic_Efficiencies_TwoEnergyOverlay.png"
        fig.savefig(output_filename, dpi=300)
        print(f"  -> Saved {output_filename}")
        plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Overlay reconstructed |t| estimators for Υ(1S), comparing 10x100 and 18x275
in each panel with the correct energy-specific selections.
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

TREE_NAME = "UpsilonTree"
EXPECTED_LUMI_INVFB = 10.0

SAMPLES = {
    "low": {
        "file": "~/merged_elecs_test_10t100_10x100.root",
        "label": "10×100 GeV",
        "campaign": "26.04.1 Campaign",
        "electron_energy": 10.0,
        "visible_xsec_fb": 104.793,
        "color": "dodgerblue",
        "w_min": 10.0,
        "w_max": 26.5,
    },
    "high": {
        "file": "~/merged_elecs_test_tuples_fix2.root",
        "label": "18×275 GeV",
        "campaign": "26.03.1 Campaign",
        "electron_energy": 18.0,
        "visible_xsec_fb": 327.584,
        "color": "darkorange",
        "w_min": 37.0,
        "w_max": 114.0,
    },
}

STATE_NORM_MODE = "target_visible_xsec"   # "target_visible_xsec" or "raw"
APPLY_SIGMA_EPZ_CUT = True
SIGMA_EPZ_HALF_WIDTH_FRACTION = 0.20

Y1S_MASS = 9.46030
Y1S_TRUTH_WINDOW = 0.20

T_BINS = np.linspace(0.0, 5.0, 51)

METHODS = [
    {
        "key": "eX",
        "branch": "reco_t_eX",
        "weight": "weight_visible",
        "label": r"$t_{eX}$",
        "xlabel": r"Reco $|t_{eX}|$ [GeV$^2$]",
    },
    {
        "key": "BABE",
        "branch": "reco_t_BABE",
        "weight": "weight_visible",
        "label": r"$t_{\rm BABE}$",
        "xlabel": r"Reco $|t_{\rm BABE}|$ [GeV$^2$]",
    },
    {
        "key": "eXBABE",
        "branch": "reco_t_eXBABE",
        "weight": "weight_visible",
        "label": r"$t_{eX{\rm BABE}}$",
        "xlabel": r"Reco $|t_{eX{\rm BABE}}|$ [GeV$^2$]",
    },
    {
        "key": "proj",
        "branch": "reco_t_proj",
        "weight": "weight_proj_visible",
        "label": r"$t_{\rm proj}$",
        "xlabel": r"Projective $|t|$ [GeV$^2$]",
    },
    {
        "key": "pT2",
        "branch": "reco_pT2",
        "weight": "weight_visible",
        "label": r"$p_{T,\Upsilon}^{2}$",
        "xlabel": r"Reco $p_{T,\Upsilon}^{2}$ [GeV$^2$]",
    },
]


def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights * weights)
    return counts, np.sqrt(variances)


def truth_1s_expr():
    return f"true_mass >= {Y1S_MASS - Y1S_TRUTH_WINDOW:.12g} && true_mass <= {Y1S_MASS + Y1S_TRUTH_WINDOW:.12g}"


def sigma_epz_info(sample_key):
    cfg = SAMPLES[sample_key]
    center = 2.0 * cfg["electron_energy"]
    half_width = SIGMA_EPZ_HALF_WIDTH_FRACTION * cfg["electron_energy"]
    return center, half_width


def sigma_epz_cut_expr(sample_key):
    center, half_width = sigma_epz_info(sample_key)
    return f"abs(sigma_epz - {center:.12g}) <= {half_width:.12g}"


def add_visible_weight_columns(df, sample_key):
    cfg = SAMPLES[sample_key]
    scale = 1.0

    if STATE_NORM_MODE == "target_visible_xsec":
        sumw = df.Filter(truth_1s_expr()).Sum("weight").GetValue()
        target_sumw = cfg["visible_xsec_fb"] * EXPECTED_LUMI_INVFB
        scale = target_sumw / sumw if sumw > 0 else 1.0
        raw_xsec = sumw / EXPECTED_LUMI_INVFB if EXPECTED_LUMI_INVFB > 0 else 0.0
        print(f"{cfg['label']} {cfg['campaign']}: raw visible 1S xsec = {raw_xsec:.3f} fb, "
              f"target = {cfg['visible_xsec_fb']:.3f} fb, scale = {scale:.6g}")
    elif STATE_NORM_MODE != "raw":
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    return (
        df.Define(f"state_norm_{sample_key}", f"{scale}")
          .Define("weight_visible", f"weight * state_norm_{sample_key}")
          .Define("weight_proj_visible", f"weight_proj * state_norm_{sample_key}")
          .Define("reco_pT2", "reco_pT * reco_pT")
    )


def base_selection(sample_key):
    cfg = SAMPLES[sample_key]
    cuts = [
        "reco_mass >= 8.0 && reco_mass <= 12.0",
        f"reco_W >= {cfg['w_min']:.12g} && reco_W <= {cfg['w_max']:.12g}",
        f"({truth_1s_expr()})",
    ]
    if APPLY_SIGMA_EPZ_CUT:
        cuts.append(sigma_epz_cut_expr(sample_key))
    return " && ".join(cuts)


def load_sample_arrays(sample_key, branches):
    cfg = SAMPLES[sample_key]
    df = ROOT.RDataFrame(TREE_NAME, cfg["file"])
    df = add_visible_weight_columns(df, sample_key)
    df_sel = df.Filter(base_selection(sample_key))
    return df_sel.AsNumpy(branches)


def draw_common_labels(ax):
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=11, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, r"$e+p\ \Upsilon(1S)\to e^+e^-$", transform=ax.transAxes,
            fontsize=10, ha="left", va="top")
    ax.text(0.04, 0.82, r"Visible $e^+e^-$ yields, $L=10~\mathrm{fb}^{-1}$", transform=ax.transAxes,
            fontsize=10, ha="left", va="top")

    cL, hwL = sigma_epz_info("low")
    cH, hwH = sigma_epz_info("high")

    lines = [
        rf"10×100, {SAMPLES['low']['campaign']}, $W_{{\gamma p}}\in[{SAMPLES['low']['w_min']:g},{SAMPLES['low']['w_max']:g}]$ GeV",
        rf"$|\Sigma(E-p_z)-{cL:g}|<{hwL:g}$ GeV",
        rf"18×275, {SAMPLES['high']['campaign']}, $W_{{\gamma p}}\in[{SAMPLES['high']['w_min']:g},{SAMPLES['high']['w_max']:g}]$ GeV",
        rf"$|\Sigma(E-p_z)-{cH:g}|<{hwH:g}$ GeV",
    ]
    y = 0.76
    for line in lines:
        ax.text(0.04, y, line, transform=ax.transAxes, fontsize=9.5, ha="left", va="top")
        y -= 0.06


def main():
    print("\n=== Υ(1S) overlay of reconstructed |t| estimators in each panel ===")
    print(f"STATE_NORM_MODE = {STATE_NORM_MODE}")
    print(f"APPLY_SIGMA_EPZ_CUT = {APPLY_SIGMA_EPZ_CUT}\n")

    branches = [
        "reco_t_eX", "reco_t_BABE", "reco_t_eXBABE", "reco_t_proj",
        "reco_pT", "reco_pT2", "weight_visible", "weight_proj_visible"
    ]
    sample_data = {k: load_sample_arrays(k, branches) for k in ["low", "high"]}

    fig, axs = plt.subplots(3, 2, figsize=(15, 16), sharex=True)
    axs = axs.flatten()

    summary_rows = []

    for i, method_cfg in enumerate(METHODS):
        ax = axs[i]
        max_density = 0.0

        for sample_key in ["low", "high"]:
            cfg = SAMPLES[sample_key]
            data = sample_data[sample_key]

            arr = data[method_cfg["branch"]]
            wgt = data[method_cfg["weight"]]

            valid = np.isfinite(arr) & np.isfinite(wgt) & (wgt > 0) & (arr >= T_BINS[0]) & (arr <= T_BINS[-1])

            if np.count_nonzero(valid) == 0:
                continue

            counts, errs = make_hist(arr[valid], T_BINS, wgt[valid])
            density = counts / np.diff(T_BINS)
            density_err = errs / np.diff(T_BINS)

            hep.histplot(density, bins=T_BINS, yerr=density_err, histtype="step",
                         color=cfg["color"], ax=ax, linewidth=2.0,
                         label=rf"{cfg['label']}")
            hep.histplot(density, bins=T_BINS, histtype="fill",
                         color=cfg["color"], alpha=0.12, ax=ax)

            max_density = max(max_density, float(np.max(density)) if len(density) else 0.0)

            summary_rows.append({
                "method": method_cfg["key"],
                "sample": sample_key,
                "label": cfg["label"],
                "campaign": cfg["campaign"],
                "sum_weight": float(np.sum(wgt[valid])),
                "n_entries": int(np.count_nonzero(valid)),
                "w_min": cfg["w_min"],
                "w_max": cfg["w_max"],
            })

        ax.set_title(method_cfg["label"], fontsize=16, pad=8)
        ax.set_xlabel(method_cfg["xlabel"], fontsize=13)
        ax.set_ylabel(r"$dN/d|t|$ [GeV$^{-2}$]", fontsize=13)
        ax.set_yscale("log")
        ax.set_xlim(T_BINS[0], T_BINS[-1])
        ax.set_ylim(0.1, max_density * 100.0 if max_density > 0 else 10.0)
        draw_common_labels(ax)
        ax.legend(frameon=False, fontsize=10, loc="upper right")

    # Hide last empty pad
    if len(axs) > len(METHODS):
        axs[-1].axis("off")

    fig.suptitle(r"Reconstructed $|t|$ estimators for $\Upsilon(1S)$, overlaid by beam energy",
                 fontsize=20, fontweight="bold", y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.985])

    out_png = "Upsilon_1S_TMethods_TwoEnergy_Overlay_visibleEE.png"
    fig.savefig(out_png, dpi=300)
    print(f"Saved {out_png}")

    import csv
    out_csv = out_png.replace(".png", "_summary.csv")
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Saved {out_csv}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import mplhep as hep
    hep.style.use(hep.style.ROOT)
except Exception:
    pass

# ==============================================================================
# 1. CONFIGURATION (Edit these parameters directly in the notebook!)
# ==============================================================================
input_file        = "~/merged_elecs_test_tuples.root"
tree_name         = "UpsilonTree"
target_state      = "all"   # Options: "1S", "2S", "3S", or "all"
eta_acceptance    = 3.5
pt_acc_min        = 0.0
electron_energy   = 18.0
sigma_half_width  = 3.6
w_min             = 20.0
lumi_invfb        = 10.0
sqrt_s            = 141.0
use_weights       = False   # Set to True if you want to apply RDataFrame weights
min_den           = 5.0     # Minimum denominator events to calculate efficiency
outdir            = "decay_lepton_eff_notebook"

# ==============================================================================
# 2. DEFINITIONS & FUNCTIONS
# ==============================================================================
UPSILON_STATES = {
    "1S": {"min_mass": 9.0,  "max_mass": 9.8,  "title": r"$\Upsilon(1S) \rightarrow e^{+}e^{-}$"},
    "2S": {"min_mass": 9.8,  "max_mass": 10.2, "title": r"$\Upsilon(2S) \rightarrow e^{+}e^{-}$"},
    "3S": {"min_mass": 10.2, "max_mass": 10.6, "title": r"$\Upsilon(3S) \rightarrow e^{+}e^{-}$"},
}

def draw_epic_labels(ax, lumi, sqrts):
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes,
            fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$",
            transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.76, rf"$\mathcal{{L}}_{{proj}} = {lumi:g}$ fb$^{{-1}}$, $\sqrt{{s}} = {sqrts:g}$ GeV",
            transform=ax.transAxes, fontsize=11, ha="left", va="top")

def weighted_efficiency(variable, pass_mask, total_mask, bins, weights=None, min_d=5.0):
    variable = np.asarray(variable)
    pass_mask = np.asarray(pass_mask, dtype=bool)
    total_mask = np.asarray(total_mask, dtype=bool)
    
    if weights is None:
        weights = np.ones_like(variable, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    v_den = variable[total_mask]
    w_den = weights[total_mask]
    v_num = variable[total_mask & pass_mask]
    w_num = weights[total_mask & pass_mask]

    den, _ = np.histogram(v_den, bins=bins, weights=w_den)
    num, _ = np.histogram(v_num, bins=bins, weights=w_num)
    den2, _ = np.histogram(v_den, bins=bins, weights=w_den * w_den)

    eff = np.full_like(den, np.nan, dtype=float)
    err = np.full_like(den, np.nan, dtype=float)
    valid = den >= min_d
    eff[valid] = num[valid] / den[valid]

    neff = np.zeros_like(den, dtype=float)
    ok = den2 > 0
    neff[ok] = den[ok] * den[ok] / den2[ok]
    valid_err = valid & (neff > 0)
    err[valid_err] = np.sqrt(np.clip(eff[valid_err] * (1.0 - eff[valid_err]), 0.0, None) / neff[valid_err])
    
    return eff, err, num, den

def integrated_eff(pass_mask, total_mask, weights=None):
    pass_mask = np.asarray(pass_mask, dtype=bool)
    total_mask = np.asarray(total_mask, dtype=bool)
    
    if weights is None:
        weights = np.ones_like(pass_mask, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
        
    den = weights[total_mask].sum()
    num = weights[total_mask & pass_mask].sum()
    den2 = (weights[total_mask] ** 2).sum()
    neff = den * den / den2 if den2 > 0 else 0.0
    
    if den <= 0:
        return np.nan, np.nan
        
    eff = num / den
    err = np.sqrt(max(eff * (1.0 - eff), 0.0) / neff) if neff > 0 else np.nan
    return 100.0 * eff, 100.0 * err

def plot_eff_panel(ax, variable, mask_acc, mask_sel, mask_gen, bins, xlabel, weights=None):
    acc_eff, acc_err, _, acc_den = weighted_efficiency(variable, mask_acc, mask_gen, bins, weights, min_den)
    sel_eff, sel_err, _, sel_den = weighted_efficiency(variable, mask_sel, mask_acc, bins, weights, min_den)
    tot_eff, tot_err, _, tot_den = weighted_efficiency(variable, mask_sel, mask_gen, bins, weights, min_den)

    acc_avg, acc_avg_err = integrated_eff(mask_acc, mask_gen, weights)
    sel_avg, sel_avg_err = integrated_eff(mask_sel, mask_acc, weights)
    tot_avg, tot_avg_err = integrated_eff(mask_sel, mask_gen, weights)

    centers = 0.5 * (bins[:-1] + bins[1:])
    ax.errorbar(centers, acc_eff, yerr=acc_err, fmt="o-", color="royalblue",
                label=rf"Geom. acceptance [{acc_avg:.1f} $\pm$ {acc_avg_err:.1f}%]", markersize=4)
    ax.errorbar(centers, sel_eff, yerr=sel_err, fmt="s-", color="darkorange",
                label=rf"Event selection after acc. [{sel_avg:.1f} $\pm$ {sel_avg_err:.1f}%]", markersize=4)
    ax.errorbar(centers, tot_eff, yerr=tot_err, fmt="^-", color="black",
                label=rf"Total event eff. [{tot_avg:.1f} $\pm$ {tot_avg_err:.1f}%]", markersize=4)

    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel("Efficiency", fontsize=14)
    ax.set_ylim(0.0, 1.45)
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
    draw_epic_labels(ax, lumi_invfb, sqrt_s)
    ax.legend(loc="lower right", frameon=False, fontsize=10)
    
    return {
        "bin_center": centers,
        "acceptance_den": acc_den,
        "selection_den_after_acceptance": sel_den,
        "total_den": tot_den,
        "acceptance_eff": acc_eff,
        "selection_eff_after_acceptance": sel_eff,
        "total_eff": tot_eff,
    }

def make_denom_plot(ax, variable, mask_gen, mask_acc, bins, xlabel, weights=None):
    if weights is None:
        weights = np.ones_like(variable, dtype=float)
        
    gen, _ = np.histogram(variable[mask_gen], bins=bins, weights=weights[mask_gen])
    acc, _ = np.histogram(variable[mask_acc], bins=bins, weights=weights[mask_acc])
    centers = 0.5 * (bins[:-1] + bins[1:])
    
    ax.step(centers, gen, where="mid", label="gen denominator")
    ax.step(centers, acc, where="mid", label="accepted denominator")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Weighted counts")
    ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=9)

# ==============================================================================
# 3. MAIN NOTEBOOK EXECUTION
# ==============================================================================
def run_notebook_analysis():
    infile = os.path.expanduser(input_file)
    os.makedirs(outdir, exist_ok=True)
    
    try:
        df = ROOT.RDataFrame(tree_name, infile)
    except Exception as e:
        print(f"Error opening file {infile}: {e}")
        return

    branches = [
        "true_mass",
        "true_ePlus_pT", "true_ePlus_eta",
        "true_decay_eMinus_pT", "true_decay_eMinus_eta",
        "reco_mass", "reco_W", "sigma_epz", "weight",
    ]
    
    print("Pulling arrays from RDataFrame into NumPy...")
    data = df.AsNumpy(branches)
    data = {k: np.asarray(v) for k, v in data.items()}
    weights = data["weight"].astype(float) if use_weights else np.ones_like(data["true_mass"], dtype=float)

    in_acc = (
        (np.abs(data["true_ePlus_eta"]) < eta_acceptance) &
        (np.abs(data["true_decay_eMinus_eta"]) < eta_acceptance) &
        (data["true_ePlus_pT"] > pt_acc_min) &
        (data["true_decay_eMinus_pT"] > pt_acc_min)
    )
    
    passed_event_selection = (
        (data["reco_mass"] >= 8.0) &
        (data["reco_mass"] <= 12.0) &
        (np.abs(data["sigma_epz"] - 2.0 * electron_energy) <= sigma_half_width) &
        (data["reco_W"] >= w_min)
    )

    states = UPSILON_STATES if target_state == "all" else {target_state: UPSILON_STATES[target_state]}
    pt_bins = np.linspace(0.0, 10.0, 40)
    eta_bins = np.linspace(-4.0, 4.0, 40)

    for state_name, cfg in states.items():
        print(f"\nProcessing {state_name}...")
        mask_state = (data["true_mass"] > cfg["min_mass"]) & (data["true_mass"] <= cfg["max_mass"])
        
        if np.sum(mask_state) == 0:
            print(f"  -> No events found for {state_name}. Skipping.")
            continue

        mask_gen = mask_state
        mask_acc = mask_state & in_acc
        mask_sel = mask_state & passed_event_selection

        # --- 1. Efficiency Dashboard ---
        fig, axs = plt.subplots(2, 2, figsize=(16, 14))
        fig.suptitle(f"{cfg['title']} | Event-level acceptance and selection efficiency",
                     fontsize=20, fontweight="bold", y=0.96)
                     
        summaries = []
        panels = [
            (axs[0, 0], data["true_decay_eMinus_pT"], pt_bins, r"True decay $e^-$ $p_T$ [GeV/$c$]"),
            (axs[0, 1], data["true_decay_eMinus_eta"], eta_bins, r"True decay $e^-$ $\eta$"),
            (axs[1, 0], data["true_ePlus_pT"], pt_bins, r"True decay $e^+$ $p_T$ [GeV/$c$]"),
            (axs[1, 1], data["true_ePlus_eta"], eta_bins, r"True decay $e^+$ $\eta$"),
        ]
        
        for ax, variable, bins, xlabel in panels:
            out = plot_eff_panel(ax, variable, mask_acc, mask_sel, mask_gen, bins, xlabel, weights=weights)
            tmp = {k: np.asarray(v) for k, v in out.items()}
            tmp["xlabel"] = np.array([xlabel] * len(tmp["bin_center"]), dtype=object)
            summaries.append(tmp)

        fig.tight_layout(rect=[0, 0, 1, 0.94])
        outpng = os.path.join(outdir, f"Upsilon_{state_name}_event_level_efficiencies.png")
        fig.savefig(outpng, dpi=250)
        plt.show() # Display inline in the notebook!
        print(f"Saved: {outpng}")

        # --- 2. Denominator Dashboard ---
        fig2, axs2 = plt.subplots(2, 2, figsize=(14, 10))
        fig2.suptitle(f"{cfg['title']} | Denominators used in efficiency plots", fontsize=18, fontweight="bold")
        
        denom_panels = [
            (axs2[0, 0], data["true_decay_eMinus_pT"], pt_bins, r"True decay $e^-$ $p_T$ [GeV/$c$]"),
            (axs2[0, 1], data["true_decay_eMinus_eta"], eta_bins, r"True decay $e^-$ $\eta$"),
            (axs2[1, 0], data["true_ePlus_pT"], pt_bins, r"True decay $e^+$ $p_T$ [GeV/$c$]"),
            (axs2[1, 1], data["true_ePlus_eta"], eta_bins, r"True decay $e^+$ $\eta$"),
        ]
        
        for ax, variable, bins, xlabel in denom_panels:
            make_denom_plot(ax, variable, mask_gen, mask_acc, bins, xlabel, weights=weights)
            
        fig2.tight_layout(rect=[0, 0, 1, 0.94])
        outpng2 = os.path.join(outdir, f"Upsilon_{state_name}_denominators.png")
        fig2.savefig(outpng2, dpi=250)
        plt.show() # Display inline in the notebook!

        # --- 3. Export CSV Data ---
        rows = []
        for s in summaries:
            for i in range(len(s["bin_center"])):
                rows.append({k: s[k][i] for k in s})
                
        csv_out = os.path.join(outdir, f"Upsilon_{state_name}_efficiency_points.csv")
        pd.DataFrame(rows).to_csv(csv_out, index=False)

# Trigger the execution
run_notebook_analysis()